# Verifiable Quantum Circuit Compilation

## v1 · Public Portfolio, Re-verified

Iteration v1 is our starting point. It takes the cheapest public circuit for each of the three tasks from one pinned
public dataset, checks every file by SHA-256, runs the **unmodified official verifiers** from the competition data at
their default shot counts, and writes `/kaggle/working/submission.zip`. No circuit here is ours. The point of this
version is a reproducible, verified baseline for our own experiments in later versions.

| Task | Circuit | Reported cost | Ratio vs reference |
|---|---|---:|---:|
| `invsqrt` | Ehan Mallick's 62-qubit *qf2* (two AND merges on yoyobar's y6225) | 100,568 | 20.65 |
| `dqi` | yoyobar's selected update of Akmal Xodarev's (busyaprime) rewrites of SomeJoint's 892-qubit decoder | 3.774e11 | 64.76 |
| `dfthcsos` | Lera's (9, 143, 40) factorization and angle loaders, Dmitrii Gluzdov's k = 0 and 14 loaders, heliosli's weights | 3.268e8 | 10.27 |

Expected geometric mean ≈ **23.95**. Costs above are the authors' reported numbers. The cells below recompute
them with this session's verifier run and a fresh salt.

**Kaggle setup:** Accelerator *None* (CPU only; kickmix does not use a GPU), Internet off. Inputs: the competition data
and the dataset [yoyobar/qcc-local-sota-20261031](https://www.kaggle.com/datasets/yoyobar/qcc-local-sota-20261031).

All credit for the circuits goes to the authors listed in **Sources and credits** at the end of this notebook.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────────────────────
VERIFY = 'full'               # 'full'  : official verifiers at their default shot counts (~15-20 CPU minutes)
                              # 'off'   : skip verification, package only (use for a quick re-submit)
SALT = 'flexonafft-v1'        # local verifier salt; the leaderboard uses its own secret salt
RELEASE_DATASET = 'yoyobar/qcc-local-sota-20261031'
RELEASE_MARKER = 'portfolio-manifest.json'     # file that identifies the release dataset's root

# Pinned bytes. A mismatch stops the notebook instead of submitting an unknown circuit.
PINNED_SHA256 = {
    'invsqrt.kmb': 'bce2948ff24e20fe59666d131f7fc4316a905103937272c52fdd39d494b6fe47',   # Ehan Mallick qf2
}

# Reference costs of the sample submission (score 1.0), as the paper's verifiers report them.
REFERENCE_COST = {
    'invsqrt': 2_076_774.233715282,          # Q 260 x C 7,987.59 (our run of the official verifier)
    'dqi': 24_439_463_583_672.93,            # Q 1,893, T ~5.725M, R(eps+) 1.1913 (as SomeJoint and busyaprime report)
    'dfthcsos': 3_355_599_588.3155365,       # lambda_eff x T x Q of the provided baseline
}
TASKS = ('invsqrt', 'dqi', 'dfthcsos')

In [ ]:
# ── Session: paths, kickmix and the official verifiers ────────────────────────────────────────
import os, sys, json, time, math, shutil, hashlib, zipfile
from pathlib import Path

NB_T0 = time.time()
ROOT = Path('/kaggle/working'); ROOT.mkdir(parents=True, exist_ok=True); os.chdir(ROOT)
INPUT = Path('/kaggle/input')
SUB = ROOT / 'submission'

def find_competition() -> Path:
    for p in (INPUT / 'competitions' / 'google-verifiable-quantum-circuit-compilation',
              INPUT / 'google-verifiable-quantum-circuit-compilation'):
        if (p / 'verifiers').is_dir():
            return p
    hits = [p.parent for p in INPUT.rglob('verify_submission_invsqrt.py')]
    if hits:
        return hits[0].parent.parent
    raise FileNotFoundError('Competition data not found: add the competition to the notebook inputs.')

def find_release() -> Path:
    hits = sorted(INPUT.rglob(RELEASE_MARKER))
    if len(hits) != 1:
        raise FileNotFoundError(f'Expected one {RELEASE_MARKER} from {RELEASE_DATASET}; found {len(hits)}.')
    return hits[0].parent

COMP, RELEASE = find_competition(), find_release()
sys.path[:0] = [str(COMP / 'kickmix'), str(COMP / 'verifiers')]
import numpy as np
import pandas as pd
import kickmix as km
from invsqrt import verify_submission_invsqrt as v_invsqrt
from dqi import verify_submission_dqi as v_dqi
from dfthcsos import verify as v_dfthcsos

print('competition data :', COMP)
print('release dataset  :', RELEASE)
print('CPU cores        :', os.cpu_count())

## 1. Assemble the portfolio

The release dataset ships a ready `submission/` tree: `invsqrt.kmb`, `dqi.kmb` and `dfthcsos/` with fifteen
`angle_k_*.kmb` loaders and `submission.npz`. Each file is copied byte for byte and hashed. The manifest is printed as
the dataset's own record of where each file came from.

In [ ]:
def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 22), b''):
            h.update(chunk)
    return h.hexdigest()

src = RELEASE / 'submission'
assert src.is_dir(), f'{src} missing: dataset layout changed'
if SUB.exists():
    shutil.rmtree(SUB)
shutil.copytree(src, SUB)

FILES = sorted(p.relative_to(SUB).as_posix() for p in SUB.rglob('*') if p.is_file())
angles = sorted(f for f in FILES if f.startswith('dfthcsos/angle_k_'))
expected = {'invsqrt.kmb', 'dqi.kmb', 'dfthcsos/submission.npz', *(f'dfthcsos/angle_k_{i}.kmb' for i in range(len(angles)))}
assert set(FILES) == expected, f'unexpected submission tree: {sorted(set(FILES) ^ expected)}'

HASHES = {f: sha256(SUB / f) for f in FILES}
for name, digest in PINNED_SHA256.items():
    assert HASHES[name] == digest, f'{name}: {HASHES[name]} differs from pinned {digest}'
for f in FILES:
    print(f'{HASHES[f][:16]}  {(SUB / f).stat().st_size / 2**20:8.2f} MiB  {f}')
print(len(FILES), 'files,', len(angles), 'angle loaders; pinned hashes match')

manifest = json.loads((RELEASE / RELEASE_MARKER).read_text())
print(json.dumps(manifest, indent=1)[:3000])

## 2. Official verification

The three verifiers run unmodified from the competition's `verifiers/` folder. `invsqrt` checks every
`x ∈ [1, 3,133,452]` 64 times; `dqi` runs 102,400 shots and prices failures through `R(ε₊)`; `dfthcsos` decodes the
angle words from the circuits and checks the factorization against the FeMoCo reference. Any `FAIL` stops the notebook.

In [ ]:
RESULTS = {}
if VERIFY == 'full':
    t0 = time.time()
    circuit = km.Circuit.from_file(str(SUB / 'invsqrt.kmb'), format='kmb')
    RESULTS['invsqrt'] = v_invsqrt.verify(circuit, v_invsqrt.ProblemSpecInvSqrt.for_n_p(10), salt=SALT)
    print(f'invsqrt   {time.time() - t0:7.0f} s'); t0 = time.time()
    circuit = km.Circuit.from_file(str(SUB / 'dqi.kmb'), format='kmb')
    RESULTS['dqi'] = v_dqi.verify(circuit, v_dqi.ProblemSpecDQI.benchmark_instance(), salt=SALT)
    del circuit
    print(f'dqi       {time.time() - t0:7.0f} s'); t0 = time.time()
    RESULTS['dfthcsos'] = v_dfthcsos.verify(str(SUB / 'dfthcsos'), salt=SALT)
    print(f'dfthcsos  {time.time() - t0:7.0f} s')
    for task in TASKS:
        assert RESULTS[task]['status'] == 'PASS', (task, RESULTS[task])
else:
    print('VERIFY = off: packaging without a verifier run')

In [ ]:
def describe(task: str, r: dict) -> str:
    if task == 'invsqrt':
        return f"Q {r['num_qubits']}, C {r['toffoli_equivalent']:.1f}, worst error/tol {r['worst_error_over_tolerance']:.5f}"
    if task == 'dqi':
        return (f"Q {r['num_qubits']}, T {r['toffoli_count']:.0f}, "
                f"{r.get('value_errors', 0)} value / {r.get('phase_errors', 0)} phase errors")
    return f"lambda_eff {r['lambda_eff']:.4f}, T {r['toffoli_count']:.0f}, Q {r['num_qubits']}"

if RESULTS:
    rows = []
    for task in TASKS:
        r = RESULTS[task]
        cost = float(r['score'])
        rows.append({'task': task, 'cost': cost, 'ratio': REFERENCE_COST[task] / cost, 'detail': describe(task, r)})
    SUMMARY = pd.DataFrame(rows).set_index('task')
    GM = math.prod(SUMMARY.ratio) ** (1 / 3)
    display(SUMMARY.style.format({'cost': '{:.6g}', 'ratio': '{:.4f}'}))
    print(f'Expected leaderboard score (geometric mean of reference / cost): {GM:.5f}')

## 3. Write the submission

The archive holds the files at its root, as the Evaluation page asks: `invsqrt.kmb`, `dqi.kmb` and `dfthcsos/`.

In [ ]:
ZIP = ROOT / 'submission.zip'
with zipfile.ZipFile(ZIP, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for f in FILES:
        z.write(SUB / f, f)
with zipfile.ZipFile(ZIP) as z:
    assert sorted(z.namelist()) == FILES
shutil.rmtree(SUB)

(ROOT / 'provenance.json').write_text(json.dumps({
    'version': 'v1_public_portfolio', 'release_dataset': RELEASE_DATASET, 'salt': SALT, 'verify': VERIFY,
    'sha256': HASHES, 'results': {t: {k: v for k, v in r.items() if isinstance(v, (int, float, str))} for t, r in RESULTS.items()},
}, indent=1))
print('SUBMISSION READY:', ZIP, f'{ZIP.stat().st_size / 2**20:.1f} MiB', '| verify:', VERIFY,
      f'| {time.time() - NB_T0:.0f} s total')

# Sources and credits

Every circuit in this submission was written by other participants and is used byte for byte from a public, Apache-2.0
dataset. The lineages below follow the credits the authors themselves give; if anyone is missing, please tell us in the
comments and we will add them.

- **Release dataset:** yoyobar, [qcc-local-sota-20261031](https://www.kaggle.com/datasets/yoyobar/qcc-local-sota-20261031),
  from [LB 22.99 | invsqrt 101 - 62 qubits](https://www.kaggle.com/code/yoyobar/lb-22-99-invsqrt-101-62-qubits).
  The same three circuits are re-verified and released by heliosli in
  [The dataset is public now. Enjoy!](https://www.kaggle.com/code/heliosli/the-dataset-is-public-now-enjoy), whose
  assembly this notebook follows.

- **invsqrt** — Ehan Mallick, [invsqrt 20.65: two 3-to-2 AND merges on y6225](https://www.kaggle.com/code/ehanmallick/invsqrt-20-65-two-3-to-2-and-merges-on-y6225)
  (two GF(2) quadratic-form windows where three ANDs become two), on yoyobar and GPT-6 Astra's 62-qubit circuit.
  Earlier lineage as the authors credit it: mrmeows / team selenium
  ([21.0 circuits](https://www.kaggle.com/code/mrmeows/21-0-circuits-let-s-all-reach-the-plateau)) → Akmal Xodarev
  (busyaprime) 63 qubits → yoyobar's cone rewrites → Ahmed Berat Özer's phase correction (as distributed in manwithacat's
  [VQCC Quantum Voodoo](https://www.kaggle.com/code/manwithacat/vqcc-quantum-voodoo)); earlier still Phuc Thai (DePie),
  Dmitrii Gluzdov, heliosli and marsyas.

- **dqi** — yoyobar's selected combination of lead-product reuse and product caching on Akmal Xodarev's (busyaprime)
  [Exact DQI Rewrites at 892 Qubits](https://www.kaggle.com/code/busyaprime/lb-23-91003-exact-dqi-rewrites-at-892-qubits)
  (flag clears by X-basis measurement, symbolic AND caching, interval allocation), which uses manwithacat's two-wire
  clean ladder ([Hedge Wizardry HW-07](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook)) and
  Ahmed Berat Özer's product-caching idea, on SomeJoint's (Agniv Banerjee's team)
  [892-qubit decoder](https://www.kaggle.com/code/somejoint/quantum-circuits-23-85-local-gm-892-qubit-dqi). SomeJoint
  credits GPT-6 Astra Pro, Claude, Ali Haydar Özdağ (flag carry), yoyobar (one-carry history) and the earlier decoder
  line: jday96314 (QuantumAI), Alex Chilton, mrmeows, manwithacat, Dmitrii Gluzdov, ViberTeam and Phuc Thai (DePie).

- **dfthcsos** — Lera (marculera), [DF-THC 10x: Compact Groups From the Hamiltonian](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian):
  the (9, 143, 40) factorization, its 8-bit angle words and the loaders k = 1 to 13; Dmitrii Gluzdov,
  [Quantum Circuit Compilation from Source](https://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source):
  the loaders k = 0 and k = 14; heliosli: the table weights `W`, `H2_I`, continuing Ahmed Berat Özer's refinement in
  [DF-THC Group Compression](https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression); assembly by
  manwithacat ([The Real $150K Was the Friends We Made](https://www.kaggle.com/code/manwithacat/the-real-150k-was-the-friends-we-made));
  Atakan Aldemir pointed heliosli to Lera's tables. Lera's own sources: Ahmed Berat Özer, somejoint, mrmeows, heliosli,
  lightningv08, kozykappa, Kozykorpesh and Sumit Chongder, Artem1410 (anazemcev), haideptry via phucthaiv02,
  Michael Rustad, k256net and Dmitrii Gluzdov.

- **Organizers:** Google Quantum AI (Tanuj Khattar, Nicholas C. Rubin, Craig Gidney, Michael Broughton, Guang Hao Low,
  Ryan Babbush) for the tasks, the verifiers and kickmix
  ([arXiv:2610.23832](https://arxiv.org/abs/2610.23832)), which keep their own terms.

The circuits keep their authors' licences; the notebook code is Apache-2.0.